# DST Fine-Tuning

Fine-tuning Llama 3.2 3B on the MLP4CS DST dataset using Unsloth + (Q)LoRA.

Task: given conversation history + user utterance → predict domain, intent, slots.

## Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Unsloth Model & (Q)LoRA

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True
# if True: QLoRA with base model in 4-bit, adapters in 16-bit (needs ~3GB VRAM)
# if False: LoRA with a base model in 16-bit (needs ~6GB VRAM)

fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Small-Instruct-2409",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",

    "unsloth/Llama-3.2-1B-bnb-4bit",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-27 13:12:43.750868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774617163.774025    1041 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774617163.781575    1041 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774617163.802157    1041 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774617163.802180    1041 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774617163.802182    1041 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none", # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## Test BEFORE Fine-Tuning

Run 3 examples from our dataset through the base model. We expect poor results as the model has never seen our task format.

In [6]:
import json
with open("/kaggle/input/datasets/georgiosxydias/mw22-finetune-data/dst_train.json", "r") as f:
    all_examples = json.load(f)

In [7]:
# Pick 3 examples to test
test_examples = all_examples[:3]
test_examples

[{'dialogue_id': 'PMUL4398.json',
  'turn_id': '0',
  'instruction': 'Extract the domain, intent, and belief state slots from the conversation.',
  'input': "\nUSER: i need a place to dine in the center thats expensive\n\nValid domains: restaurant, hotel\nValid intents: find_hotel, book_hotel, find_restaurant, book_restaurant\nValid slots: hotel-area, hotel-pricerange, hotel-type, hotel-stars, hotel-internet, hotel-parking, hotel-name, hotel-bookday, hotel-bookpeople, hotel-bookstay, restaurant-area, restaurant-pricerange, restaurant-food, restaurant-name, restaurant-bookday, restaurant-bookpeople, restaurant-booktime\n\nDOMAIN: <domain>\nINTENT: <intent>\nSLOTS: <slot1=value1, slot2=value2> or SLOTS: none\nRules:\n- ONLY extract slot values EXPLICITLY stated by the user.\n- If user expresses no preference for a specific attribute (e.g. 'any food', 'no particular type', 'doesn't matter') → identify WHICH slot they refer to and extract slot=dontcare.\n  Example: 'no particular food type

In [8]:
def run_inference(model, tokenizer, example):
    messages = [
        {"role": "user", "content": f"{example['instruction']}\n\n{example['input']}"}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True, do_sample=False) # do_sample=False=temperature=0.0:always picks the highest probability token
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return decoded.split("assistant")[-1].strip()

In [9]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("RESULTS BEFORE FINE-TUNING")
print("=" * 60)
for i, ex in enumerate(test_examples):
    pred = run_inference(model, tokenizer, ex)
    print(f"\n>>> Example {i+1} (turn {ex['turn_id']})")
    print(f"\nINPUT:\n{ex['input']}")
    print(f"\nEXPECTED:\n{ex['output']}")
    print(f"\nPREDICTION:\n{pred}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


RESULTS BEFORE FINE-TUNING

>>> Example 1 (turn 0)

INPUT:

USER: i need a place to dine in the center thats expensive

Valid domains: restaurant, hotel
Valid intents: find_hotel, book_hotel, find_restaurant, book_restaurant
Valid slots: hotel-area, hotel-pricerange, hotel-type, hotel-stars, hotel-internet, hotel-parking, hotel-name, hotel-bookday, hotel-bookpeople, hotel-bookstay, restaurant-area, restaurant-pricerange, restaurant-food, restaurant-name, restaurant-bookday, restaurant-bookpeople, restaurant-booktime

DOMAIN: <domain>
INTENT: <intent>
SLOTS: <slot1=value1, slot2=value2> or SLOTS: none
Rules:
- ONLY extract slot values EXPLICITLY stated by the user.
- If user expresses no preference for a specific attribute (e.g. 'any food', 'no particular type', 'doesn't matter') → identify WHICH slot they refer to and extract slot=dontcare.
  Example: 'no particular food type' → restaurant-food=dontcare
  Example: 'any area is fine' → hotel-area=dontcare
  Example: 'no preference on pr

## Data Preparation


Format our JSON into Alpaca-style text: `### Instruction / ### Input / ### Response`.

In [10]:
from datasets import Dataset

def format_example(ex: dict) -> dict:
    text = (
        f"### Instruction:\n{ex['instruction']}\n\n"
        f"### Input:\n{ex['input']}\n\n"
        f"### Response:\n{ex['output']}"
    )
    return {"text": text}

dataset = Dataset.from_list([format_example(e) for e in all_examples])
print(f"Loaded {len(dataset)} DST training examples.")

Loaded 10846 DST training examples.


In [11]:
dataset

Dataset({
    features: ['text'],
    num_rows: 10846
})

In [12]:
print("Sample formatted text (first example):")
print(dataset[0]["text"])

Sample formatted text (first example):
### Instruction:
Extract the domain, intent, and belief state slots from the conversation.

### Input:

USER: i need a place to dine in the center thats expensive

Valid domains: restaurant, hotel
Valid intents: find_hotel, book_hotel, find_restaurant, book_restaurant
Valid slots: hotel-area, hotel-pricerange, hotel-type, hotel-stars, hotel-internet, hotel-parking, hotel-name, hotel-bookday, hotel-bookpeople, hotel-bookstay, restaurant-area, restaurant-pricerange, restaurant-food, restaurant-name, restaurant-bookday, restaurant-bookpeople, restaurant-booktime

DOMAIN: <domain>
INTENT: <intent>
SLOTS: <slot1=value1, slot2=value2> or SLOTS: none
Rules:
- ONLY extract slot values EXPLICITLY stated by the user.
- If user expresses no preference for a specific attribute (e.g. 'any food', 'no particular type', 'doesn't matter') → identify WHICH slot they refer to and extract slot=dontcare.
  Example: 'no particular food type' → restaurant-food=dontcare


## Train

In [13]:
# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

GPU = Tesla T4. Max memory = 14.563 GB.
3.051 GB of memory reserved before training.


In [14]:
optim = "adamw_8bit" if load_in_4bit else "adamw_torch"
print(f"Using {optim} optimizer.")

Using adamw_8bit optimizer.


In [15]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,  # full training: sees all 11,686 examples, e.g., for 3 epochs × 3 times (~4hrs on T4)
        # max_steps = 60,  # quick test: stops after 60 steps regardless of dataset size (~4min on T4)
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = optim,
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "dst_outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/10846 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [16]:
trainer.train_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10846
})

In [17]:
# Train
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,846 | Num Epochs = 1 | Total steps = 1,356
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.056300
20,0.830100
30,0.421100
40,0.380600
50,0.386800
60,0.376200
70,0.308600
80,0.343600
90,0.311400
100,0.342200


In [18]:
# GPU memory after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for LoRA = {used_memory_for_lora} GB.")


142.08 minutes used for training.
Peak reserved memory = 4.207 GB.
Peak reserved memory for LoRA = 1.156 GB.


## Test AFTER Fine-Tuning

Same 3 examples as before. Compare predicted vs expected output.

In [19]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("RESULTS AFTER FINE-TUNING")
print("=" * 60)
for i, ex in enumerate(test_examples):
    pred = run_inference(model, tokenizer, ex)
    print(f"\n>>> Example {i+1} (turn {ex['turn_id']})")
    print(f"\nINPUT:\n{ex['input']}")
    print(f"\nEXPECTED:\n{ex['output']}")
    print(f"\nPREDICTION:\n{pred}")

RESULTS AFTER FINE-TUNING

>>> Example 1 (turn 0)

INPUT:

USER: i need a place to dine in the center thats expensive

Valid domains: restaurant, hotel
Valid intents: find_hotel, book_hotel, find_restaurant, book_restaurant
Valid slots: hotel-area, hotel-pricerange, hotel-type, hotel-stars, hotel-internet, hotel-parking, hotel-name, hotel-bookday, hotel-bookpeople, hotel-bookstay, restaurant-area, restaurant-pricerange, restaurant-food, restaurant-name, restaurant-bookday, restaurant-bookpeople, restaurant-booktime

DOMAIN: <domain>
INTENT: <intent>
SLOTS: <slot1=value1, slot2=value2> or SLOTS: none
Rules:
- ONLY extract slot values EXPLICITLY stated by the user.
- If user expresses no preference for a specific attribute (e.g. 'any food', 'no particular type', 'doesn't matter') → identify WHICH slot they refer to and extract slot=dontcare.
  Example: 'no particular food type' → restaurant-food=dontcare
  Example: 'any area is fine' → hotel-area=dontcare
  Example: 'no preference on pri

## Save LoRA adapter

In [21]:
# Save locally 
model.save_pretrained("dst_lora")
tokenizer.save_pretrained("dst_lora")

# Zip into Kaggle's output folder 
import shutil
shutil.make_archive("/kaggle/working/dst_lora", "zip", "dst_lora")

# Optional: push to HuggingFace Hub
# model.push_to_hub("YOUR_HF_USERNAME/mlp4cs-dst-lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_HF_USERNAME/mlp4cs-dst-lora", token="YOUR_HF_TOKEN")

'/kaggle/working/dst_lora.zip'